# M5A1 - Inspeção Visual de Itens em Esteira de Manufatura

Na prática de hoje vamos refinar um modelo para a tarefa de inspeção visual.

Esse notebook está estruturado da seguinte forma.

- Introdução
- Carregar Base de Dados
- Refinar Modelo
- Próximos passos
- Atividade Complementares

## Introdução

Instalação para os que ainda não possuem a biblioteca instalada.

In [1]:
%pip install torch torchvision huggingface_hub ultralytics

Note: you may need to restart the kernel to use updated packages.


Importar as bibliotecas

In [2]:
import yaml
import os

from ultralytics import YOLO
from huggingface_hub import snapshot_download
from pathlib import Path
import shutil
from IPython.display import Video

## Carregar Base de Dados

A primeira tarefa para refinar um modelo é criar a base de dados.

Referência: https://huggingface.co/datasets/johnatanvq/fruits-dataset

In [3]:
# 1) Baixar somente o subdiretório fruitsData/
local_repo_dir = snapshot_download(
    repo_id="johnatanvq/fruits-dataset",
    repo_type="dataset",
    allow_patterns=["fruitsData/**"],  # baixa só essa pasta
)

print("Arquivos baixados em:", local_repo_dir)

# 2) Mover/copiar para uma pasta final estilo ImageFolder (se quiser customizar o caminho)
src = Path(local_repo_dir) / "fruitsData"
dst = Path("data/fruits")  # pasta final onde você quer o ImageFolder

# Copiando arquivos para dst.
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)

print("ImageFolder pronto em:", dst)


Fetching 322 files:   0%|          | 0/322 [00:00<?, ?it/s]

Arquivos baixados em: /Users/joseph-cbp/.cache/huggingface/hub/datasets--johnatanvq--fruits-dataset/snapshots/7cf7e5de00d186e99fe46357a80be02f3e110a34
ImageFolder pronto em: data/fruits


In [ ]:
def create_data_yaml(path_to_classes_txt, path_to_data_yaml):
  # Lê o arquivos "classes.txt".
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Cria o dicionário a ser salvo.
  data = {
      'path': 'data/fruits',
      'train': 'images',
      'val': 'images',
      'nc': number_of_classes,
      'names': classes
  }

  # Escreve o arquivo YAML.
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Chama a função.
create_data_yaml("data/fruits/classes.txt", "yolo_train.yaml")

Created config file at yolo_train.yaml


In [5]:
# Carrega o modelo pré-treinado.
model = YOLO("yolo11n.pt")

# Treina o modelo utilizando as informações do arquivo YAML.
# Definimos também a quantidade de épocas, o batch, e o tamanho das imagens.
results = model.train(data="./yolo_train.yaml", project="praticas/modulo_5/aula_1", epochs=10, batch=2, imgsz=480)

New https://pypi.org/project/ultralytics/8.4.67 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.66 🚀 Python-3.14.3 torch-2.12.0 CPU (Apple M2 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_train.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset

In [6]:
# Video pode ser necessário alterar o path.
Video("video.mov")


In [7]:
model.predict("video.mov", save=True, project="praticas/modulo_5/aula_1")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/239) /Users/joseph-cbp/Documents/Visão Computacional/VC - Mod05/VC_M5A1 - Inspeção Visual de Itens em Esteira de Manufatura/video.mov: 288x480 3 apples, 1 orange, 26.4ms
video 1/1 (frame 2/239) /Users/joseph-cbp/Documents/Visão Computacional/VC - Mod05/VC_M5A1 - Inspeção Visual de Itens em Esteira de Manufatura/video.mov: 288x480 3 apples, 1 orange, 22.4ms
video 1/1 (frame 3/239) /Users/joseph-cbp/Documents/Visão Computacional/V

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'apple', 1: 'carrot', 2: 'orange'}
 obb: None
 orig_img: array([[[ 97,  87,  65],
         [ 97,  87,  65],
         [ 97,  87,  65],
         ...,
         [187, 179, 152],
         [192, 182, 158],
         [192, 182, 158]],
 
        [[ 97,  87,  65],
         [ 97,  87,  65],
         [ 97,  87,  65],
         ...,
         [187, 179, 152],
         [192, 182, 158],
         [192, 182, 158]],
 
        [[ 97,  87,  65],
         [ 97,  87,  65],
         [ 97,  87,  65],
         ...,
         [187, 179, 152],
         [192, 182, 158],
         [190, 181, 157]],
 
        ...,
 
        [[ 48,  52,  24],
         [ 48,  52,  24],
         [ 48,  52,  22],
         ...,
         [ 25, 133, 203],
         [ 26, 134, 204],
         [ 26, 134, 204]],
 
        [[ 48,  52,  24],
         [ 48,  52,  24],
         [ 48,  52,  22],
       

In [ ]:
Video("../../runs/detect/praticas/modulo_5/aula_1/predict/video.mp4")

ValueError: To embed videos, you must pass embed=True (this may make your notebook files huge)
Consider passing Video(url='...')

## Próximos Passos e Referências

Nas próximas práticas vamos continuar trabalhando com problemas reais que envolvem Visão Computacional.

Uma lista não exaustiva de referências segue:

- https://docs.ultralytics.com/modes/train/
- https://docs.ultralytics.com/modes/predict/
- https://huggingface.co/datasets/johnatanvq/fruits-dataset
- https://huggingface.co/datasets
- https://pytorch.org/
- https://docs.pytorch.org/vision/main/models.html
- https://opencv.org/
- https://learnopencv.com/blogs/
- https://pyimagesearch.com/

## Atividades Complementares (Opcional)

- [ ] Tente alterar a base de dados e veja se o modelo continua funcionando?
- [ ] Tente alterar alguns hiperparâmetros de treinamento, batch e resolução da imagens e veja como isso altera os resultados.